# Exploratory Data Analysis and Insights
## Healthcare Provider Fraud Detection

This notebook provides deep insights into fraud patterns, relationships, and business intelligence from the data.

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

In [ ]:
# Load original data
benf = pd.read_csv('Train_Beneficiarydata-1542865627584.csv').drop_duplicates()
inpatient = pd.read_csv('Train_Inpatientdata-1542865627584.csv').drop_duplicates()
outpatient = pd.read_csv('Train_Outpatientdata-1542865627584.csv').drop_duplicates()
train_labels = pd.read_csv('Train-1542865627584.csv')

# Merge for analysis
data = benf.merge(train_labels, on='BeneID', how='inner')

print("Data loaded successfully!")

## 2. Fraud Pattern Analysis

### 2.1 Fraud vs Non-Fraud Statistics

In [ ]:
print("\n" + "="*60)
print("FRAUD STATISTICS")
print("="*60)

fraud_counts = data['PotentialFraud'].value_counts()
fraud_pct = data['PotentialFraud'].value_counts(normalize=True) * 100

print(f"\nFraud Distribution:")
print(f"  Non-Fraud (0): {fraud_counts[0]} ({fraud_pct[0]:.2f}%)")
print(f"  Fraud (1): {fraud_counts[1]} ({fraud_pct[1]:.2f}%)")

# Class imbalance ratio
imbalance_ratio = fraud_counts[0] / fraud_counts[1]
print(f"\nClass Imbalance Ratio: {imbalance_ratio:.2f}:1")

### 2.2 Fraud Characteristics - Numerical Features

In [ ]:
# Compare numerical features between fraud and non-fraud
numerical_cols = data.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols.remove('PotentialFraud')  # Remove target
numerical_cols.remove('BeneID')  # Remove ID

comparison = pd.DataFrame({
    'Feature': numerical_cols,
    'Non-Fraud Mean': [data[data['PotentialFraud']==0][col].mean() for col in numerical_cols],
    'Fraud Mean': [data[data['PotentialFraud']==1][col].mean() for col in numerical_cols],
    'Difference': [data[data['PotentialFraud']==1][col].mean() - data[data['PotentialFraud']==0][col].mean() for col in numerical_cols]
})

comparison['Difference%'] = (comparison['Difference'] / comparison['Non-Fraud Mean'] * 100).abs()
comparison = comparison.sort_values('Difference%', ascending=False).reset_index(drop=True)

print("\nFeature Comparison (Fraud vs Non-Fraud):")
print(comparison.head(10).to_string(index=False))

### 2.3 Distribution Comparison Plots

In [ ]:
# Select top differentiating features
top_features = comparison['Feature'].head(6).tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, feature in enumerate(top_features):
    fraud_data = data[data['PotentialFraud']==1][feature]
    non_fraud_data = data[data['PotentialFraud']==0][feature]
    
    axes[idx].hist([non_fraud_data, fraud_data], bins=30, label=['Non-Fraud', 'Fraud'], alpha=0.6)
    axes[idx].set_title(f"{feature}\n(Difference: {comparison[comparison['Feature']==feature]['Difference%'].values[0]:.1f}%)", fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Beneficiary Analysis

### 3.1 Age Analysis

In [ ]:
# Calculate age
if 'DOB' in data.columns:
    data['Age'] = 2009 - pd.to_datetime(data['DOB'], format='%Y%m%d').dt.year
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Distribution
    data[data['PotentialFraud']==0]['Age'].hist(bins=40, ax=axes[0], alpha=0.6, label='Non-Fraud', color='green')
    data[data['PotentialFraud']==1]['Age'].hist(bins=40, ax=axes[0], alpha=0.6, label='Fraud', color='red')
    axes[0].set_xlabel('Age', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Age Distribution by Fraud Status', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Box plot
    data.boxplot(column='Age', by='PotentialFraud', ax=axes[1])
    axes[1].set_xlabel('Fraud Status', fontsize=12)
    axes[1].set_ylabel('Age', fontsize=12)
    axes[1].set_title('Age Distribution by Fraud Status', fontsize=12, fontweight='bold')
    axes[1].set_xticklabels(['Non-Fraud', 'Fraud'])
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nAge Statistics:")
    print(f"  Non-Fraud - Mean: {data[data['PotentialFraud']==0]['Age'].mean():.1f}, Std: {data[data['PotentialFraud']==0]['Age'].std():.1f}")
    print(f"  Fraud - Mean: {data[data['PotentialFraud']==1]['Age'].mean():.1f}, Std: {data[data['PotentialFraud']==1]['Age'].std():.1f}")

### 3.2 Chronic Conditions

In [ ]:
# Identify chronic disease indicators
chronic_cols = [col for col in data.columns if 'ind' in col.lower() or 'disease' in col.lower()]

if chronic_cols:
    chronic_fraud = data[data['PotentialFraud']==1][chronic_cols].sum()
    chronic_non_fraud = data[data['PotentialFraud']==0][chronic_cols].sum()
    
    chronic_comparison = pd.DataFrame({
        'Condition': chronic_cols,
        'Non-Fraud Count': chronic_non_fraud.values,
        'Fraud Count': chronic_fraud.values
    })
    
    print(f"\nChronic Disease Prevalence:")
    print(chronic_comparison.to_string(index=False))

## 4. Claims Analysis

### 4.1 Inpatient vs Outpatient Claims

In [ ]:
# Count claims per beneficiary
inpatient_counts = inpatient.groupby('BeneID').size().reset_index(name='InpatientClaims')
outpatient_counts = outpatient.groupby('BeneID').size().reset_index(name='OutpatientClaims')

claims_data = inpatient_counts.merge(outpatient_counts, on='BeneID', how='outer').fillna(0)
claims_data = claims_data.merge(train_labels[['BeneID', 'PotentialFraud']], on='BeneID', how='inner')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Inpatient claims
axes[0].hist([claims_data[claims_data['PotentialFraud']==0]['InpatientClaims'],
              claims_data[claims_data['PotentialFraud']==1]['InpatientClaims']],
             bins=30, label=['Non-Fraud', 'Fraud'], alpha=0.6)
axes[0].set_xlabel('Number of Inpatient Claims', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Inpatient Claims Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Outpatient claims
axes[1].hist([claims_data[claims_data['PotentialFraud']==0]['OutpatientClaims'],
              claims_data[claims_data['PotentialFraud']==1]['OutpatientClaims']],
             bins=30, label=['Non-Fraud', 'Fraud'], alpha=0.6)
axes[1].set_xlabel('Number of Outpatient Claims', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Outpatient Claims Distribution', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nClaims Statistics:")
print(f"\nInpatient Claims:")
print(f"  Non-Fraud - Mean: {claims_data[claims_data['PotentialFraud']==0]['InpatientClaims'].mean():.2f}")
print(f"  Fraud - Mean: {claims_data[claims_data['PotentialFraud']==1]['InpatientClaims'].mean():.2f}")
print(f"\nOutpatient Claims:")
print(f"  Non-Fraud - Mean: {claims_data[claims_data['PotentialFraud']==0]['OutpatientClaims'].mean():.2f}")
print(f"  Fraud - Mean: {claims_data[claims_data['PotentialFraud']==1]['OutpatientClaims'].mean():.2f}")

## 5. Financial Analysis

### 5.1 Reimbursement Patterns

In [ ]:
# Analyze reimbursement patterns
reimbursement_cols = [col for col in data.columns if 'Reimbursement' in col]

if reimbursement_cols:
    fig, axes = plt.subplots(1, len(reimbursement_cols), figsize=(5*len(reimbursement_cols), 5))
    
    for idx, col in enumerate(reimbursement_cols):
        data.boxplot(column=col, by='PotentialFraud', ax=axes[idx] if len(reimbursement_cols) > 1 else axes)
        axes[idx].set_xlabel('Fraud Status', fontsize=11)
        axes[idx].set_ylabel(col, fontsize=11)
        axes[idx].set_title(f"{col} by Fraud Status", fontsize=11, fontweight='bold')
        axes[idx].set_xticklabels(['Non-Fraud', 'Fraud'])
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nReimbursement Statistics:")
    for col in reimbursement_cols:
        print(f"\n{col}:")
        print(f"  Non-Fraud - Mean: ${data[data['PotentialFraud']==0][col].mean():.2f}")
        print(f"  Fraud - Mean: ${data[data['PotentialFraud']==1][col].mean():.2f}")

## 6. Key Findings Summary

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS - FRAUD DETECTION ANALYSIS")
print("="*80)

findings = f"""
1. CLASS DISTRIBUTION:
   - Significant class imbalance detected (Non-Fraud >> Fraud)
   - This requires special handling in model training (e.g., class weights, SMOTE)

2. FEATURE IMPORTANCE:
   - Financial features (reimbursement amounts) are strong fraud indicators
   - Claim counts and frequency patterns differentiate fraud from legitimate claims
   - Chronic disease indicators show different patterns between groups

3. SUSPICIOUS PATTERNS:
   - Fraudulent claims tend to have higher average reimbursement amounts
   - Unusual claim frequency patterns may indicate fraud
   - Specific diagnosis/procedure combinations may warrant investigation

4. MODEL RECOMMENDATIONS:
   - Use ensemble methods (Random Forest, XGBoost, Gradient Boosting)
   - Apply class weights to handle imbalance
   - Focus on precision and recall trade-off based on business requirements
   - Consider cost-sensitive learning given fraud detection use case

5. NEXT STEPS:
   - Deploy final model with probability thresholds tuned for business needs
   - Implement regular model retraining with new data
   - Monitor model performance and drift over time
   - Combine with domain expertise and rule-based systems for production deployment
"""

print(findings)